# 242 - convex NMF at every K, both feature sets

**Run 240, 241 and 242 in parallel** - three kernels, one per algorithm. They share the dataset cache read-only and write to separate run directories and separate sweep files, so they cannot collide.

**Before this:** `python rebuild_concat_cache.py --apply` must have finished. It is the only serial step.

**After all three:** run `243_cluster_statistics.ipynb`.

| | |
|---|---|
| method | convex NMF |
| feature sets | `concat_hg`, `concat_rawds` (gated cohort, one set of electrodes) |
| K | 5 .. 30 |
| cNMF fit iterations | 1000 final, 300 inside cross-validation |


In [ ]:
import os, sys, json, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
assert (ROOT / 'functions').exists(), f'run this from 02_FBM_Clustering, not {ROOT}'
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'functions'))
import lf_concat as CC, lf_cluster_run as R, lf_runs as LR, config as cfg

SCRIPT_NAME = '242_cluster_cnmf.ipynb'
METHOD      = 'cnmf'
METHOD_LBL  = 'convex NMF'
CACHE_DIR   = ROOT / 'outputs/_dataset/concat_source_v3'
FEATURE_SETS= ['concat_hg', 'concat_rawds']
K_RANGE     = [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
N_ITER      = 1000          # convex NMF fit iterations (final fits)
RANDOM_STATE= cfg.RANDOM_STATE

INPUT_DIR = Path(r'\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = (ROOT.parent / '01_FBM_Analysis' / 'outputs' / '04_ersp_LM_RAWONLY').resolve()

print('method      :', METHOD_LBL)
print('feature sets:', FEATURE_SETS)
print('K range     :', K_RANGE[0], '..', K_RANGE[-1], f'({len(K_RANGE)} values)')
print('cache       :', CACHE_DIR)

## 1 - Load the shared cohort

Built once by `rebuild_concat_cache.py`. This notebook only reads it.

In [ ]:
# The cache MUST already exist. rebuild_concat_cache.py builds it once, serially.
# Building it here would mean three kernels racing to write the same files, and
# prepare_dataset's cache key carries no patient list, so a half-written cache would
# be picked up as a hit by whichever kernel arrived second.
assert CACHE_DIR.exists() and (CACHE_DIR / 'X_3d.npy').exists(), (
    f'{CACHE_DIR} missing - run:  python rebuild_concat_cache.py --apply')

df_contacts, X_concat = CC.build_concat_dataset(
    INPUT_DIR, conditions=('audio','picture','reading'),
    require_high_activity=True, cache_dir=CACHE_DIR, verbose=True)

X = {'concat_hg':    CC.concat_hg_features(X_concat, hg_band=(70.0,150.0), fmax=500.0),
     'concat_rawds': CC.concat_rawds_features(X_concat, n_blocks=3, fmax_hz=500.0)}

print(f'\ncohort: {len(df_contacts)} electrodes · '
      f'{df_contacts.patient_id.nunique()} patients')
for k_, v in X.items():
    print(f'  {k_:<14} {v.shape}')

## 2 - Fit convex NMF at every K

In [ ]:
# convex NMF: run_decomposition creates the run, sweep_decomposition fills in
# loadings_by_k / components_by_k / cluster_labels_by_k for EVERY K.
def sh(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    r = subprocess.run([sys.executable, *[str(c) for c in cmd]],
                       cwd=str(ROOT), env={**os.environ, 'PYTHONIOENCODING':'utf-8'})
    assert r.returncode == 0, f'failed: {cmd}'

for fs in FEATURE_SETS:
    t0 = time.time()
    sh(['run_decomposition.py', '--feature-set', fs])
    sh(['sweep_decomposition.py', '--feature-set', fs,
        '--ks', *[str(k) for k in K_RANGE], '--n-iter', str(N_ITER)])
    print(f'{fs} done in {time.time()-t0:.0f}s')

for fs in FEATURE_SETS:
    rd = LR.newest_run('cnmf', fs)
    have = sorted(int(p.stem[3:]) for p in (rd/'loadings_by_k').glob('G_k*.npy'))
    print(f'{fs:<14} {rd.name}   loadings at K {have[0]}..{have[-1]} ({len(have)})')
    assert set(K_RANGE).issubset(have), f'{fs}: missing loadings_by_k'

## 3 - Held-out variance, K=5..30

Bi-cross-validation: a block of rows AND a block of columns is held out, so an extra component has to earn its place. Home space only - that is what the peak-K decision reads.

In [ ]:
# Held-out variance, K=5..30, HOME SPACE only, bi-cross-validated.
# --tag keeps this kernel's output in its own file so the three notebooks running in
# parallel never write to the same CSV.
cmd = [sys.executable, 'make_heldout_variance.py',
       '--from-cache', str(CACHE_DIR),
       '--feature-set', *FEATURE_SETS,
       '--method', METHOD, '--tag', METHOD,
       '--ks', *[str(k) for k in K_RANGE],
       '--spaces', 'home', '--n-iter', '300']
print('$', ' '.join(cmd), flush=True)
r = subprocess.run(cmd, cwd=str(ROOT), env={**os.environ, 'PYTHONIOENCODING':'utf-8'})
assert r.returncode == 0

pk = pd.read_csv(ROOT/'outputs'/'clustering'/'bsf_comparison'/f'heldout_peaks_{METHOD}.csv')
display(pk)

## 4 - Done

In [ ]:
print('DONE:', METHOD_LBL)
print('When ALL THREE of 240 / 241 / 242 have finished, run 243.')